# Day 3 — Single-agent baseline

First, validate a manually constructed report against collected evidence.
Then replace manual report construction with a tool-using agent.

The football dataset contains fictional test data.

In [1]:
import json

from aegis.schemas import (
    ComparisonRow,
    FinalReport,
    ResearchBundle,
    validate_report_against_bundle,
)
from aegis.tools.evidence import lookup_evidence

In [2]:
# Retrieve scoring evidence using our existing deterministic tool.
# Each lookup returns both seasons for the requested player.
lookups = [
    lookup_evidence(
        subject=subject,
        factor="scoring",
        dataset_ref="football_demo_v1",
    )
    for subject in ["Player A", "Player B"]
]

# Both players belong to the same source dataset.
# Deduplicate sources by ID before constructing the bundle.
sources_by_id = {}
evidence_by_id = {}

for lookup in lookups:
    if lookup.source is None or not lookup.evidence:
        raise ValueError(
            f"Expected fixture evidence, received: {lookup.reason}"
        )

    sources_by_id[lookup.source.source_id] = lookup.source

    for item in lookup.evidence:
        evidence_by_id[item.evidence_id] = item

# ResearchBundle checks that the evidence references valid sources.
bundle = ResearchBundle(
    sources=list(sources_by_id.values()),
    evidence=list(evidence_by_id.values()),
)

print(f"Sources: {len(bundle.sources)}")
print(f"Evidence items: {len(bundle.evidence)}")

Sources: 1
Evidence items: 4


In [3]:
# Select the requested season explicitly so we don't mix periods.
selected_evidence = [
    item
    for item in bundle.evidence
    if item.time_period == "2025"
]

comparison_rows = []

for item in selected_evidence:
    # The fixture tool preserves its selected fields as JSON.
    observation = json.loads(item.excerpt)

    comparison_rows.append(
        ComparisonRow(
            subject=item.subject,
            factor="scoring",
            value=observation["goals"],
            unit="goals",
            evidence_ids=[item.evidence_id],
        )
    )

# This is a handwritten report, not an LLM-generated response.
# The numerical rows come from retrieved fixture evidence.
report = FinalReport(
    summary=(
        "In the fictional 2025 Demo League fixture, "
        "Player A recorded 14 goals and Player B recorded 11."
    ),
    mode="compare",
    subjects=["Player A", "Player B"],
    comparison_rows=comparison_rows,
    ranking=[],
    findings=[],
    limitations=[
        "Synthetic test data; these are not real player statistics.",
        "Comparison covers only goals in the 2025 Demo League.",
        "Goal totals do not account for differences in playing time.",
    ],
    citations=[source.source_id for source in bundle.sources],
    status="completed",
)

# Check that every referenced evidence item and source exists.
validated_report = validate_report_against_bundle(report, bundle)

print(validated_report.model_dump_json(indent=2))
print("PASS: handwritten report validates against fixture evidence")

{
  "summary": "In the fictional 2025 Demo League fixture, Player A recorded 14 goals and Player B recorded 11.",
  "mode": "compare",
  "subjects": [
    "Player A",
    "Player B"
  ],
  "comparison_rows": [
    {
      "subject": "Player A",
      "factor": "scoring",
      "value": 14.0,
      "unit": "goals",
      "evidence_ids": [
        "ev_42aa13f70da1973f10be907aa173e5b20984b0b3f94dd6d6671edb4fa10a6519"
      ]
    },
    {
      "subject": "Player B",
      "factor": "scoring",
      "value": 11.0,
      "unit": "goals",
      "evidence_ids": [
        "ev_ca42f0862bb7a926e9972fb79e7e95b69a9621a0d45352a94d1c624fc84ef972"
      ]
    }
  ],
  "ranking": [],
  "findings": [],
  "limitations": [
    "Synthetic test data; these are not real player statistics.",
    "Comparison covers only goals in the 2025 Demo League.",
    "Goal totals do not account for differences in playing time."
  ],
  "citations": [
    "src_football_demo_v1"
  ],
  "status": "completed"
}
PASS: handwri